<a href="https://colab.research.google.com/github/Traiba/aurora-siger/blob/main/analysis-aurora-siger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install -q -U google-genai

import time
import random
import pandas as pd
import os
from google import genai
from google.colab import files

GOOGLE_API_KEY = ''

def analise_ia_missao(telemetria, relatorio_falhas):
    """
    Utiliza a nova biblioteca google-genai para o parecer do Diretor de Voo.
    """
    try:
        client = genai.Client(api_key=GOOGLE_API_KEY)

        status = "CRÍTICO" if relatorio_falhas else "OPERACIONAL"

        prompt = f"""
        Você é o Diretor de Voo da Missão Aurora-SIGER.
        Analise a telemetria: {telemetria}
        Falhas: {relatorio_falhas}
        Veredito: {status}

        Dê um parecer técnico de 3 linhas com tom sério de agência espacial.
        """

        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )

        return response.text
    except Exception as e:
        return f"Erro de conexão com o Diretor de Voo: {e}"

def carregar_arquivo_local():
    print("Por favor, faça o upload do arquivo 'dataset_telemetria_marte.csv'")
    uploaded = files.upload()
    for nome_arquivo in uploaded.keys():
        print(f' Arquivo "{nome_arquivo}" carregado!')
        return nome_arquivo
    return None

# ─────────────────────────────────────────────
#  3. FUNÇÕES DE EXIBIÇÃO E AUXILIARES
# ─────────────────────────────────────────────
def exibir_sucesso():
    print("\n 🚀 UM PEQUENO PASSO PARA UM HOMEM, UM SALTO GIGANTE PARA O BRASIL!")
    print(" 🚩 ESTAÇÃO MARCIANA ALCANÇADA - POUSO REALIZADO EM MARTE COM SUCESSO.")
    print(" [MISSÃO AURORA-SIGER CONCLUÍDA!]")

def exibir_falha(motivos_falha):
    print("\n ❌ MISSÃO ABORTADA: FALHAS DETECTADAS")
    for motivo in motivos_falha:
        print(f" - {motivo}")
    print(" 🆘 PROCEDIMENTOS DE EMERGÊNCIA ATIVADOS. CÁPSULA EJETADA.")
    print(" [FALHA NA JORNADA PARA MARTE]")

def realizar_analise_energetica(carga_percentual):
    capacidade_total = 1000  # kWh
    consumo_decolagem = 300  # kWh
    perdas_percentual = 0.05
    energia_disponivel = capacidade_total * (carga_percentual / 100)
    perdas = energia_disponivel * perdas_percentual
    energia_real = energia_disponivel - perdas
    energia_restante = energia_real - consumo_decolagem
    return {
        "disponivel": energia_disponivel,
        "perdas": perdas,
        "real": energia_real,
        "restante": energia_restante,
        "seguro": energia_restante > 0
    }

# ─────────────────────────────────────────────
#  4. LÓGICA DO MINI-GAME
# ─────────────────────────────────────────────
def iniciar_mini_game(telemetria_atual):
    print(f"\n{'='*20} DILEMA NO ESPAÇO PROFUNDO {'='*20}")

    telemetria_pos_minigame = telemetria_atual.copy()

    dilemas = [
        {
            "pergunta": "Uma tempestade solar massiva está vindo! O que fazer?",
            "opcoes": [
                "1- Ativar escudos magnéticos (Gasta energia, protege integridade)",
                "2- Manter curso e confiar na blindagem (Risco de danos, economiza energia)"
            ],
            "consequencia1": {"nivel_energia": -15, "integridade_estrutural": 0, "msg": "Escudos ativados. Energia drenada, mas integridade mantida."},
            "consequencia2": {"nivel_energia": 0, "integridade_estrutural": 1, "msg": "Radiação solar atingiu os sistemas! Danos estruturais detectados."}
        },
        {
            "pergunta": "O sistema de Oxigênio está com um pequeno vazamento. O que fazer?",
            "opcoes": [
                "1- Selar o setor de carga (Reduz O2 disponível, mas evita perda total)",
                "2- Enviar drone para reparo externo (Gasta bateria extra, mantém O2)"
            ],
            "consequencia1": {"o2_pct_global": -20, "msg": "Setor selado. O2 reduzido, mas vazamento contido."},
            "consequencia2": {"nivel_energia": -10, "msg": "Drone realizou o reparo. Sistemas gastaram bateria extra."}
        }
    ]

    dilema = random.choice(dilemas)
    print(f"\n ALERTA: {dilema['pergunta']}")
    for opt in dilema['opcoes']: print(f"   {opt}")

    while True:
        escolha = input("Sua escolha (1 ou 2): ")
        if escolha in ["1", "2"]: break
        else: print("Opção inválida.")

    res = dilema["consequencia1"] if escolha == "1" else dilema["consequencia2"]
    print(f"\n RESULTADO: {res['msg']}")

    for key, value in res.items():
        if key in telemetria_pos_minigame:
            if key in ["nivel_energia", "nivel_combustivel"]:
                telemetria_pos_minigame[key] = max(0.0, telemetria_pos_minigame[key] + value)
            else:
                telemetria_pos_minigame[key] = value
        elif key == "o2_pct_global":
            global o2_pct_global
            o2_pct_global = max(0.0, o2_pct_global + value)

    return telemetria_pos_minigame

# ─────────────────────────────────────────────
#  5. ALGORITMO DE VERIFICAÇÃO GO/NO-GO
# ─────────────────────────────────────────────
def verificar_go_no_go(telemetria):
    relatorio_falhas = []

    if not (5.0 <= telemetria.get("temperatura_interna", 0) <= 40.0):
        relatorio_falhas.append(f"Temperatura Interna fora da faixa segura.")

    if telemetria.get("integridade_estrutural", 0) != 0:
        relatorio_falhas.append("Integridade Estrutural comprometida.")

    if telemetria.get("nivel_energia", 0) < 100.0:
        relatorio_falhas.append(f"Nível de Energia insuficiente ({telemetria.get('nivel_energia', 0)}%).")

    if not (98.0 <= telemetria.get("pressao_tanques_percentual", 0) <= 102.0):
        relatorio_falhas.append(f"Pressão dos Tanques fora da margem.")

    if telemetria.get("nivel_combustivel", 0) < 100.0:
        relatorio_falhas.append(f"Combustível abaixo de 100%.")

    analise_energia = realizar_analise_energetica(telemetria.get("nivel_energia", 0))
    if not analise_energia["seguro"]:
        relatorio_falhas.append(f"Cálculo energético indica risco de pane.")

    if not relatorio_falhas:
        return "PRONTO PARA DECOLAR", []
    else:
        return "DECOLAGEM ABORTADA", relatorio_falhas

# ─────────────────────────────────────────────
#  6. EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────
def simulacao_aurora_siger():
    nome_csv = carregar_arquivo_local()
    if not nome_csv: return

    try:
        df = pd.read_csv(nome_csv)
    except Exception as e:
        print(f"Erro: {e}"); return

    global o2_pct_global
    o2_pct_global = 100.0

    random_row = df.sample(n=1).iloc[0]
    telemetria_atual = random_row.to_dict()

    print(f"\n{'='*30} DADOS DE TELEMETRIA {'='*30}")
    for k, v in telemetria_atual.items():
        print(f" - {k}: {v}")

    status_inicial, falhas_iniciais = verificar_go_no_go(telemetria_atual)

    # CHAMADA DA IA PARA PARECER TÉCNICO
    print(f"\n{'-'*20} RELATÓRIO DO DIRETOR DE VOO (IA) {'-'*20}")
    print(analise_ia_missao(telemetria_atual, falhas_iniciais))
    print(f"{'-'*60}\n")

    if status_inicial == "PRONTO PARA DECOLAR":
        print("SISTEMAS NOMINAIS. INICIANDO JORNADA...")
        telemetria_final = iniciar_mini_game(telemetria_atual)

        status_pos, falhas_pos = verificar_go_no_go(telemetria_final)
        if o2_pct_global < 50: falhas_pos.append("Oxigênio Crítico!")

        # Segunda análise da IA após o evento aleatório
        print(f"\n{'-'*20} ANÁLISE PÓS-EVENTO (IA) {'-'*20}")
        print(analise_ia_missao(telemetria_final, falhas_pos))
        print(f"{'-'*60}\n")

        if status_pos == "PRONTO PARA DECOLAR" and not falhas_pos:
            exibir_sucesso()
        else:
            exibir_falha(falhas_pos)
    else:
        exibir_falha(falhas_iniciais)

if __name__ == "__main__":
    simulacao_aurora_siger()

Por favor, faça o upload do arquivo 'dataset_telemetria_marte.csv'


Saving dataset_telemetria_marte.csv to dataset_telemetria_marte (9).csv
 Arquivo "dataset_telemetria_marte (9).csv" carregado!

============================== DADOS DE TELEMETRIA ==============================
 - temperatura_interna: 47.94
 - temperatura_externa: 18.18
 - integridade_estrutural: 0
 - nivel_energia: 100.0
 - pressao_tanques_percentual: 99.59
 - status_modulos: 0
 - velocidade_vento: 9.27
 - umidade_relativa: 58.29
 - visibilidade: 32.69
 - nivel_combustivel: 100.0
 - velocidade_comunicacao: 12.25
 - angulo_fase: 36.05
 - status_final: DECOLAGEM ABORTADA

-------------------- RELATÓRIO DO DIRETOR DE VOO (IA) --------------------
Erro de conexão com o Diretor de Voo: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota 

In [9]:
!pip install -q -U google-generativeai